# 10 — Propensity-Score Matching (PSM)

Estimate the Average Treatment Effect on the Treated (ATT) of AI assistance
using nearest-neighbour propensity-score matching.

**Pipeline**
1. Estimate propensity scores via logistic regression
2. Match treated to control units (1:1, with caliper)
3. Assess covariate balance before vs. after matching
4. Estimate ATT on each outcome
5. Run placebo test and compute E-value for sensitivity

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data.generate import generate_synthetic_conversations
from src.analysis.psm import estimate_propensity_score, match_nearest_neighbor
from src.analysis.balance import balance_table
from src.sensitivity.robustness import placebo_test, e_value

sns.set_theme(style='whitegrid')
%matplotlib inline

COVARIATES = ['issue_severity', 'customer_tenure', 'time_of_day', 'agent_experience']
OUTCOMES   = ['resolution_time', 'satisfaction_score', 'escalated']
TREATMENT  = 'ai_assisted'

## 1. Load data

In [ ]:
CSV = '../data/synthetic_conversations.csv'
if os.path.exists(CSV):
    df = pd.read_csv(CSV, parse_dates=['created_at'])
else:
    df = generate_synthetic_conversations(n=2000, seed=42)

print(f'{len(df)} rows | treated={df[TREATMENT].sum()} | control={( ~df[TREATMENT].astype(bool)).sum()}')

## 2. Estimate propensity scores

In [ ]:
ps, model = estimate_propensity_score(df, treatment_col=TREATMENT, covariates=COVARIATES)
df['propensity_score'] = ps

print('Logistic regression coefficients:')
for name, coef in zip(COVARIATES, model.coef_[0]):
    print(f'  {name:25s} {coef:+.4f}')

In [ ]:
# Propensity score overlap (common support check)
fig, ax = plt.subplots(figsize=(8, 4))
for treat, grp in df.groupby(TREATMENT):
    label = 'AI assisted' if treat else 'Human only'
    grp['propensity_score'].hist(ax=ax, alpha=0.5, bins=40, density=True, label=label)
ax.set_xlabel('Propensity score')
ax.set_title('Common support — propensity score distributions')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Match treated to control

In [ ]:
matched = match_nearest_neighbor(df, ps=ps, treatment_col=TREATMENT, caliper=0.05)

n_treated = matched[TREATMENT].sum()
n_control = (~matched[TREATMENT].astype(bool)).sum()
print(f'Matched sample: {len(matched)} rows | treated={n_treated} | control={n_control}')

## 4. Covariate balance before vs. after matching

In [ ]:
bt_before = balance_table(df, treatment_col=TREATMENT, covariates=COVARIATES)
bt_after  = balance_table(matched, treatment_col=TREATMENT, covariates=COVARIATES)

balance_comparison = bt_before[['covariate', 'smd']].rename(columns={'smd': 'smd_before'}).merge(
    bt_after[['covariate', 'smd']].rename(columns={'smd': 'smd_after'}),
    on='covariate'
)
balance_comparison['smd_before'] = balance_comparison['smd_before'].abs()
balance_comparison['smd_after']  = balance_comparison['smd_after'].abs()
balance_comparison.round(3)

In [ ]:
# Love plot
fig, ax = plt.subplots(figsize=(7, 4))
y = range(len(balance_comparison))
ax.scatter(balance_comparison['smd_before'], y, marker='o', label='Before matching', zorder=3)
ax.scatter(balance_comparison['smd_after'],  y, marker='D', label='After matching',  zorder=3)
ax.set_yticks(list(y))
ax.set_yticklabels(balance_comparison['covariate'])
ax.axvline(0.1, color='red', linestyle='--', label='|SMD| = 0.1')
ax.set_xlabel('|Standardised Mean Difference|')
ax.set_title('Love plot — covariate balance')
ax.legend()
plt.tight_layout()
plt.show()

## 5. ATT estimates on matched sample

In [ ]:
results = []
for outcome in OUTCOMES:
    treated_vals = matched.loc[matched[TREATMENT] == 1, outcome]
    control_vals = matched.loc[matched[TREATMENT] == 0, outcome]
    att = treated_vals.mean() - control_vals.mean()
    t_stat, p_val = stats.ttest_ind(treated_vals, control_vals)
    results.append({'outcome': outcome, 'ATT': att, 't_stat': t_stat, 'p_value': p_val})

att_df = pd.DataFrame(results).set_index('outcome')
att_df.round(4)

In [ ]:
# Visualise matched outcomes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, outcome in zip(axes, OUTCOMES):
    for treat, grp in matched.groupby(TREATMENT):
        label = 'AI assisted' if treat else 'Human only'
        grp[outcome].hist(ax=ax, alpha=0.5, bins=25, density=True, label=label)
    ax.set_title(outcome)
    ax.legend(fontsize=8)
plt.suptitle('Outcome distributions — matched sample', y=1.02)
plt.tight_layout()
plt.show()

## 6. Sensitivity analysis

In [ ]:
# Placebo test on matched sample
for outcome in OUTCOMES:
    orig, null_effects, p = placebo_test(matched, TREATMENT, outcome, n_runs=200, seed=0)
    print(f'{outcome:25s}  observed={orig:+.3f}  placebo p={p:.3f}')

In [ ]:
# E-value for resolution_time (convert mean difference to approximate risk ratio)
mean_control_rt = matched.loc[matched[TREATMENT] == 0, 'resolution_time'].mean()
att_rt = att_df.loc['resolution_time', 'ATT']
# Approximate RR: ratio of means (human-only / AI-assisted)
rr = mean_control_rt / (mean_control_rt + att_rt)
ev = e_value(rr)
print(f'Approx RR (human vs AI) for resolution_time: {rr:.3f}')
print(f'E-value: {ev:.3f}  — an unmeasured confounder would need to associate {ev:.1f}x with both treatment and outcome to explain away the effect.')